# NYXVAULT — временный CPU inference server

Этот ноутбук запускает настоящую ONNX-модель в удалённой среде Colab CPU. Телефон получает только текст чата: веса и токенизатор скачиваются непосредственно в Colab через HTTP Range-запросы.

**Важно:** Colab и временный туннель не являются постоянным хостингом. Не закрывайте runtime во время тестирования. Ячейка сама проверяет `/health` и делает пробную генерацию.

In [ ]:
# This runs the NYXVAULT API on the remote Colab CPU, not on your phone.
# Keep the Colab runtime connected while using the website.

import os, re, sys, time, secrets, subprocess
from pathlib import Path

REPO_DIR = Path("/content/Nyxvault")
LOG_DIR = Path("/content/nyxvault-server-logs")
LOG_DIR.mkdir(exist_ok=True)

# 1) Get the prepared server code.
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "nyxvault-cpu-server",
        "https://github.com/Phantom220-hub/Nyxvault.git", str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "nyxvault-cpu-server"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "nyxvault-cpu-server"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

SERVER_DIR = REPO_DIR / "server"

# 2) Install CPU-only inference dependencies.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(SERVER_DIR / "requirements.txt")
], check=True)

# 3) Create a private API key inside this Colab runtime.
API_TOKEN = secrets.token_urlsafe(32)
os.environ["NYXVAULT_API_TOKEN"] = API_TOKEN
os.environ["NYXVAULT_MODEL_DIR"] = "/content/nyxvault-model"

# 4) Start the real ONNX inference server. It downloads only ONNX + tokenizer
# from the GitHub Release using HTTP byte ranges, directly to Colab.
server_log = open(LOG_DIR / "api.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(SERVER_DIR),
    env=os.environ.copy(),
    stdout=server_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

import requests
headers = {"Authorization": f"Bearer {API_TOKEN}"}
print("Starting NYXVAULT on the Colab CPU. Model setup can take a few minutes...")
health = None
for attempt in range(180):
    if server.poll() is not None:
        server_log.flush()
        print((LOG_DIR / "api.log").read_text(errors="replace")[-12000:])
        raise RuntimeError("NYXVAULT server stopped during startup.")
    try:
        response = requests.get("http://127.0.0.1:8000/health", headers=headers, timeout=3)
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        pass
    if attempt % 10 == 0:
        print(f"Waiting for model initialization... ({attempt * 2}s)")
    time.sleep(2)

if health is None:
    server.terminate()
    server_log.flush()
    print((LOG_DIR / "api.log").read_text(errors="replace")[-12000:])
    raise TimeoutError("NYXVAULT model did not become ready within 6 minutes.")

print("Model status:", health)
print((LOG_DIR / "api.log").read_text(errors="replace")[-5000:])

# 5) Install Cloudflare's temporary tunnel client in the remote Colab runtime.
cloudflared = Path("/usr/local/bin/cloudflared")
if not cloudflared.exists():
    subprocess.run([
        "bash", "-lc",
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"
    ], check=True)

tunnel_log_path = LOG_DIR / "tunnel.log"
tunnel_log = open(tunnel_log_path, "w")
tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

public_url = None
for attempt in range(90):
    if tunnel.poll() is not None:
        tunnel_log.flush()
        print(tunnel_log_path.read_text(errors="replace")[-8000:])
        raise RuntimeError("Cloudflare tunnel stopped.")
    tunnel_log.flush()
    log_text = tunnel_log_path.read_text(errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if not public_url:
    print(tunnel_log_path.read_text(errors="replace")[-8000:])
    raise TimeoutError("Cloudflare did not return a temporary public URL.")

# 6) Verify that the public URL reaches the real model server.
check = requests.get(public_url + "/health", headers=headers, timeout=30)
check.raise_for_status()
print("\nPUBLIC SERVER IS READY")
print("Health check:", check.json())

# Run one real generation so we know inference works, not just that the API starts.
test = requests.post(
    public_url + "/generate",
    headers=headers,
    json={
        "prompt": "Пользователь: Напиши короткое приветствие.\nNYXVAULT:",
        "max_new_tokens": 12,
        "temperature": 0.0,
        "top_k": 40,
        "stop": ["\nПользователь:"]
    },
    timeout=180,
)
test.raise_for_status()
print("Real model test:", test.json())

print("\nCopy these two values into the Lovable project's SERVER-SIDE Secrets.")
print("NYXVAULT_BACKEND_URL =", public_url)
print("NYXVAULT_BACKEND_TOKEN =", API_TOKEN)
print("\nDo not paste the token into ChatGPT or GitHub. Keep this Colab runtime running while testing.")
